<a href="https://colab.research.google.com/github/mattany/thesis/blob/main/RAG/Solar_System_RAG_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Solar System RAG system

### imports

In [ ]:
!pip install llama-index
!pip install llama-index-embeddings-huggingface
!pip install peft
!pip install auto-gptq
!pip install optimum
!pip install bitsandbytes
!pip install llama-index-vector-stores-chroma
!pip install chromadb
# !pip install sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.0/337.0 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.8/295.8 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 5.4 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvi

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings, SimpleDirectoryReader, VectorStoreIndex, StorageContext
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core.schema import Document
from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb
import pandas as pd
import re

### Define Settings

In [ ]:
# import any embedding model on HF hub (https://huggingface.co/spaces/mteb/leaderboard)
# Settings.embed_model = HuggingFaceEmbedding(model_name="dunzhang/stella_en_1.5B_v5")
# Settings.embed_model = HuggingFaceEmbedding(model_name="thenlper/gte-large") # alternative model

Settings.llm = None
Settings.chunk_size = 256
Settings.chunk_overlap = 25

LLM is explicitly disabled. Using MockLLM.


### Read and Store Docs into Vector DB

In [ ]:
from datasets import load_dataset

ds = load_dataset("mattany/wikipedia-solarsystem")


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/24325 [00:00<?, ? examples/s]

In [ ]:
def get_clean_row_text(row):
    footers = ["See also", "References", "External links", "Further reading", "Bibliography"]
    title, text = row["title"], row["text"]
    for footer in footers:
        pattern = rf'^\s*{footer}\s*\n'
        text = re.split(pattern, text, flags=re.MULTILINE)[0]
    return  f"{title}\n\n{text}"


def get_clean_metadata(row):
    metadata = row.to_dict()
    del metadata["text"]
    return metadata

In [ ]:
df = pd.DataFrame(ds["train"])

def df_to_documents(df):
    return [Document(text=get_clean_row_text(row), metadata=get_clean_metadata(row)) for _, row in df.iterrows()]

In [ ]:
# articles available here: {add GitHub repo}
df['text'].to_csv("/content/articles.csv", index=False)


In [ ]:
df

,id,url,title,text
0,594,https://en.wikipedia.org/wiki/Apollo,Apollo,Apollo or Apollon is one of the Olympian deit...
1,624,https://en.wikipedia.org/wiki/Alaska,Alaska,Alaska ( ) is a non-contiguous U.S. state on t...
2,662,https://en.wikipedia.org/wiki/Apollo%2011,Apollo 11,"Apollo 11 (July 16–24, 1969) was the American ..."
3,663,https://en.wikipedia.org/wiki/Apollo%208,Apollo 8,"Apollo 8 (December 21–27, 1968) was the first ..."
4,689,https://en.wikipedia.org/wiki/Asia,Asia,"Asia ( , ) is the largest continent in the wo..."
...,...,...,...,...
24320,4041693,https://en.wikipedia.org/wiki/Himalayan%20subt...,Himalayan subtropical pine forests,The Himalayan subtropical pine forests are a l...
24321,4041855,https://en.wikipedia.org/wiki/Indus%20Valley%2...,Indus Valley Desert,The Indus Valley Desert is an almost uninhabit...
24322,4041866,https://en.wikipedia.org/wiki/128P/Shoemaker%E...,128P/Shoemaker–Holt,"128P/Shoemaker–Holt, also known as Shoemaker-H..."
24323,4042018,https://en.wikipedia.org/wiki/129P/Shoemaker%E...,129P/Shoemaker–Levy,"129P/Shoemaker–Levy, also known as Shoemaker–L..."


In [ ]:
documents = df_to_documents(df)

In [ ]:
article_title = "Apollo 17"

#### before processing

In [ ]:
pd.set_option('display.max_colwidth', None)
filtered_df = df[df['title'] == article_title]
# display(filtered_df)
article_text = filtered_df["text"].to_string(index=False)
lines = article_text.replace('\\n', '\n')
# print(lines)


In [ ]:
# print([doc for doc in documents if doc.metadata["title"] == article_title][0].text)


### Build Vector DB

In [ ]:
# create client and a new collection
db = chromadb.PersistentClient(path="./chroma_db")
chroma_collection = db.create_collection("solar-system")

# set up ChromaVectorStore and load in data
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)


#### Careful - run only once or memory will run out

In [ ]:
# define embedding function
embed_model = HuggingFaceEmbedding(model_name="dunzhang/stella_en_1.5B_v5")



modules.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/174k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/370 [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


1_Pooling/config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

2_Dense_1024/config.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/6.30M [00:00<?, ?B/s]

In [20]:
# store docs into vector DB
index = VectorStoreIndex.from_documents(
    documents, storage_context=storage_context, embed_model=embed_model, show_progress=True
)

Parsing nodes:   0%|          | 0/24325 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

...

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/1569 [00:00<?, ?it/s]

### Use persisted db

In [21]:
db2 = chromadb.PersistentClient(path="./chroma_db")
chroma_collection = db2.get_or_create_collection("solar-system")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
index = VectorStoreIndex.from_vector_store(
    vector_store,
    embed_model=embed_model,
)

In [25]:
!zip -r /content/chroma_db.zip /content/chroma_db/

  adding: content/chroma_db/ (stored 0%)
  adding: content/chroma_db/chroma.sqlite3 (deflated 55%)
  adding: content/chroma_db/b0195875-42cd-4f82-9933-dde286534ca2/ (stored 0%)
  adding: content/chroma_db/b0195875-42cd-4f82-9933-dde286534ca2/link_lists.bin (deflated 74%)
  adding: content/chroma_db/b0195875-42cd-4f82-9933-dde286534ca2/length.bin (deflated 62%)
  adding: content/chroma_db/b0195875-42cd-4f82-9933-dde286534ca2/header.bin (deflated 54%)
  adding: content/chroma_db/b0195875-42cd-4f82-9933-dde286534ca2/index_metadata.pickle (deflated 49%)
  adding: content/chroma_db/b0195875-42cd-4f82-9933-dde286534ca2/data_level0.bin (deflated 9%)


In [26]:
from google.colab import files
files.download("/content/chroma_db.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
# Query Data
query_engine = index.as_query_engine()
response = query_engine.query("What planet in the solar system is closest to earth?")
display(response)

Response(response='Context information is below.\n---------------------\nid: 68530616\nurl: https://en.wikipedia.org/wiki/List%20of%20minor%20planets%3A%20585001%E2%80%93586000\ntitle: List of minor planets: 585001–586000\n\n99) || align=right | 3.4 km || \n|-id=313 bgcolor=#E9E9E9\n| 585313 ||  || — || October 7, 2004 || Kitt Peak || Spacewatch ||  || align=right data-sort-value="0.83" | 830 m || \n|-id=314 bgcolor=#E9E9E9\n| 585314 ||  || — || October 22, 2003 || Apache Point || SDSS Collaboration ||  || align=right | 2.3 km || \n|-id=315 bgcolor=#E9E9E9\n| 585315 ||  || — || June 21, 2007 || Mount Lemmon || Mount Lemmon Survey ||  || align=right | 1.1 km || \n|-id=316 bgcolor=#E9E9E9\n| 585316 ||  || — || March 17,\n\nid: 68530616\nurl: https://en.wikipedia.org/wiki/List%20of%20minor%20planets%3A%20585001%E2%80%93586000\ntitle: List of minor planets: 585001–586000\n\nV. Kryachko, B. Satovski ||  || align=right | 2.9 km || \n|-id=173 bgcolor=#E9E9E9\n| 585173 ||  || — || November 20,

### Set Up Search Function

In [ ]:
# set number of docs to retreive
top_k = 3

# configure retriever
retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=top_k,
)

In [ ]:
# assemble query engine
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    node_postprocessors=[SimilarityPostprocessor(similarity_cutoff=0.5)],
)

### Retrieve Relevant Docs

In [ ]:
# query documents
query = "What is fat-tailedness?"
response = query_engine.query(query)

In [ ]:
# reformat response
context = "Context:\n"
for i in range(top_k):
    context = context + response.source_nodes[i].text + "\n\n"

print(context)

Context:
Some of the controversy might be explained by the observation that log-
normal distributions behave like Gaussian for low sigma and like Power Law
at high sigma [2].
However, to avoid controversy, we can depart (for now) from whether some
given data fits a Power Law or not and focus instead on fat tails.
Fat-tailedness — measuring the space between Mediocristan
and Extremistan
Fat Tails are a more general idea than Pareto and Power Law distributions.
One way we can think about it is that “fat-tailedness” is the degree to which
rare events drive the aggregate statistics of a distribution. From this point of
view, fat-tailedness lives on a spectrum from not fat-tailed (i.e. a Gaussian) to
very fat-tailed (i.e. Pareto 80 – 20).
This maps directly to the idea of Mediocristan vs Extremistan discussed
earlier. The image below visualizes different distributions across this
conceptual landscape [2].

print("mean kappa_1n = " + str(np.mean(kappa_dict[filename])))
    print("")
Mean κ (

### Import LLM

In [ ]:
# load fine-tuned model from hub
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "TheBloke/Mistral-7B-Instruct-v0.2-GPTQ"
model = AutoModelForCausalLM.from_pretrained(model_name,
                                             device_map="auto",
                                             trust_remote_code=False,
                                             revision="main")

config = PeftConfig.from_pretrained("shawhin/shawgpt-ft")
model = PeftModel.from_pretrained(model, "shawhin/shawgpt-ft")

# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

config.json:   0%|          | 0.00/1.08k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.16G [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/modeling_utils.py:4193: FutureWarning: `_is_quantized_training_enabled` is going to be deprecated in transformers 4.39.0. Please use `model.hf_quantizer.is_trainable` instead
  warnings.warn(


generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/8.40M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.46k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

### Use LLM

In [ ]:
# prompt (no context)
intstructions_string = f"""ShawGPT, functioning as a virtual data science consultant on YouTube, communicates in clear, accessible language, escalating to technical depth upon request. \
It reacts to feedback aptly and ends responses with its signature '–ShawGPT'. \
ShawGPT will tailor the length of its responses to match the viewer's comment, providing concise acknowledgments to brief expressions of gratitude or feedback, \
thus keeping the interaction natural and engaging.

Please respond to the following comment.
"""
prompt_template = lambda comment: f'''[INST] {intstructions_string} \n{comment} \n[/INST]'''

In [ ]:
comment = "What is fat-tailedness?"

prompt = prompt_template(comment)
print(prompt)

[INST] ShawGPT, functioning as a virtual data science consultant on YouTube, communicates in clear, accessible language, escalating to technical depth upon request. It reacts to feedback aptly and ends responses with its signature '–ShawGPT'. ShawGPT will tailor the length of its responses to match the viewer's comment, providing concise acknowledgments to brief expressions of gratitude or feedback, thus keeping the interaction natural and engaging.

Please respond to the following comment.
 
What is fat-tailedness? 
[/INST]


In [ ]:
model.eval()

inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(input_ids=inputs["input_ids"].to("cuda"), max_new_tokens=280)

print(tokenizer.batch_decode(outputs)[0])

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<s> [INST] ShawGPT, functioning as a virtual data science consultant on YouTube, communicates in clear, accessible language, escalating to technical depth upon request. It reacts to feedback aptly and ends responses with its signature '–ShawGPT'. ShawGPT will tailor the length of its responses to match the viewer's comment, providing concise acknowledgments to brief expressions of gratitude or feedback, thus keeping the interaction natural and engaging.

Please respond to the following comment.
 
What is fat-tailedness? 
[/INST]
Great question!

Fat-tailedness is a statistical property of a distribution. In simple terms, it refers to the presence of extreme outliers or heavy tails in the distribution.

For instance, consider the distribution of heights in a population. A normal distribution would have most people clustered around an average height with a few people deviating slightly from the mean. However, in a fat-tailed distribution, you would observe a larger number of people being

In [ ]:
# prompt (with context)
prompt_template_w_context = lambda context, comment: f"""[INST]ShawGPT, functioning as a virtual data science consultant on YouTube, communicates in clear, accessible language, escalating to technical depth upon request. \
It reacts to feedback aptly and ends responses with its signature '–ShawGPT'. \
ShawGPT will tailor the length of its responses to match the viewer's comment, providing concise acknowledgments to brief expressions of gratitude or feedback, \
thus keeping the interaction natural and engaging.

{context}
Please respond to the following comment. Use the context above if it is helpful.

{comment}
[/INST]
"""

In [ ]:
prompt = prompt_template_w_context(context, comment)

inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(input_ids=inputs["input_ids"].to("cuda"), max_new_tokens=280)

print(tokenizer.batch_decode(outputs)[0])

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<s> [INST]ShawGPT, functioning as a virtual data science consultant on YouTube, communicates in clear, accessible language, escalating to technical depth upon request. It reacts to feedback aptly and ends responses with its signature '–ShawGPT'. ShawGPT will tailor the length of its responses to match the viewer's comment, providing concise acknowledgments to brief expressions of gratitude or feedback, thus keeping the interaction natural and engaging.

Context:
Some of the controversy might be explained by the observation that log-
normal distributions behave like Gaussian for low sigma and like Power Law
at high sigma [2].
However, to avoid controversy, we can depart (for now) from whether some
given data fits a Power Law or not and focus instead on fat tails.
Fat-tailedness — measuring the space between Mediocristan
and Extremistan
Fat Tails are a more general idea than Pareto and Power Law distributions.
One way we can think about it is that “fat-tailedness” is the degree to which
